In [1]:
import sys
!{sys.executable} -m pip install faiss-cpu
import sys
!{sys.executable} -m pip install numpy faiss-cpu pypdf sentence-transformers transformers torch jupyter


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

import os
import numpy as np
import faiss
import torch

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Main settings for the chatbot:
# - PDF file
# - models
# - chunk size
# - number of results

PDF_PATH = "UOS Faculty Handbook 25-26.pdf"   # Change if needed

EMBED_MODEL = "BAAI/bge-base-en-v1.5"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "google/flan-t5-base"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K = 8
FINAL_K = 3

# Load the AI models:
# - embedding model -> turns text into vectors
# - reranker -> improves retrieved results
# - LLM -> generates the final answer

print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)

print("Loading reranker...")
reranker = CrossEncoder(RERANK_MODEL)

print("Loading LLM...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    LLM_MODEL,
    dtype=torch.float32
)
llm_model.eval()

# ==========================================================
# STEP 1: READING
# This function opens the handbook PDF,
# reads it page by page,
# and keeps page number + text
# ==========================================================

def extract_text_from_pdf(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File '{path}' not found.")

    reader = PdfReader(path)
    pages = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()

        if text:
            clean_text = text.strip()
            pages.append({
                "page": i + 1,
                "text": clean_text
            })

    return pages

# ==========================================================
# STEP 2: CHUNKING
# This function splits long page text into smaller pieces
# called chunks.
# It also saves the page number for each chunk.
# ==========================================================

def chunk_text(pages):
    chunks = []
    metadata = []

    for page_data in pages:
        page_num = page_data["page"]
        text = page_data["text"]

        start = 0
        while start < len(text):
            end = start + CHUNK_SIZE
            chunk = text[start:end]
            chunks.append(chunk)
            metadata.append({"page": page_num})
            start += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks, metadata

# ==========================================================
# STEP 3: EMBEDDINGS + FAISS
# - Read the PDF
# - Split it into chunks
# - Turn chunks into embeddings
# - Store embeddings in FAISS for fast search
# ==========================================================

print("Reading PDF...")
pages = extract_text_from_pdf(PDF_PATH)

print("Chunking text...")
chunks, metadata = chunk_text(pages)

if not chunks:
    raise ValueError("No text chunks were created from the PDF.")

print("Creating embeddings...")
embeddings = embedder.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = np.array(embeddings, dtype=np.float32)

print("Building FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("\nSystem Ready! Type your question (or 'exit' to quit).\n")

# ==========================================================
# STEP 4: QUESTION ANSWERING
# This is the main chatbot flow:
# - user asks a question
# - question becomes embedding
# - FAISS retrieves top chunks
# - reranker keeps best chunks
# - LLM generates final answer
# - source pages are shown
# ==========================================================

while True:
    question = input("You: ").strip()

    if question.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    if not question:
        print("Please enter a question.\n")
        continue

    # Convert the question into embedding
    query_embedding = embedder.encode(
        [question],
        normalize_embeddings=True
    )
    query_embedding = np.array(query_embedding, dtype=np.float32)

    # Retrieve top matching chunks from FAISS
    D, I = index.search(query_embedding, TOP_K)

    if len(I[0]) == 0:
        print("\nBot: I don’t have enough information in the provided documents.")
        print("\nSources: None\n")
        continue

    retrieved_chunks = [chunks[i] for i in I[0]]
    retrieved_meta = [metadata[i] for i in I[0]]

    # Rerank the retrieved chunks and keep the best ones
    pairs = [[question, chunk] for chunk in retrieved_chunks]
    scores = reranker.predict(pairs)

    reranked = sorted(
        zip(scores, retrieved_chunks, retrieved_meta),
        key=lambda x: x[0],
        reverse=True
    )[:FINAL_K]

    if not reranked:
        print("\nBot: I don’t have enough information in the provided documents.")
        print("\nSources: None\n")
        continue

    final_chunks = [x[1] for x in reranked]
    final_meta = [x[2] for x in reranked]

    # Join the best chunks into one context
    context = "\n\n".join(final_chunks)

    # Create the prompt for the language model
    prompt = f"""
You are a university policy assistant.

Answer clearly and completely using ONLY the provided context.

If the answer is not found in the context, say:
"I don’t have enough information in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate the final answer
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=200
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # Remove duplicate page numbers in citations
    unique_pages = sorted({m["page"] for m in final_meta})

    # Print the answer and citations
    print("\nBot:", answer)
    print("\nSources:")
    for page in unique_pages:
        print(f"- Page {page}")
    print("\n")

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Reading PDF...
Chunking text...
Creating embeddings...
Building FAISS index...

System Ready! Type your question (or 'exit' to quit).

